In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os

In [ ]:
input_dir = '/kaggle/input/listings'  

files = os.listdir(input_dir)

# Print the list of files
print(files)

In [ ]:
import pandas as pd
import json
import os
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)


input_dir = '/kaggle/input/listings'  
json_files = os.listdir(input_dir)  

json_data = []

def filter_by_language_tag(data):
    # Check if any list has a dictionary with 'language_tag' starting with 'en_'
    for key in data:
        if isinstance(data[key], list):  # If the value is a list (like brand, bullet_point)
            for item in data[key]:
                if isinstance(item, dict) and item.get('language_tag', '').startswith('en_'):
                    return True  # If any matching language_tag found
    return False  # No matching language_tag found

# Iterate over all JSON files in the listings directory
for json_file in json_files:
    file_path = os.path.join(input_dir, json_file)
    
    # Open the file and read the JSON data
    with open(file_path, 'r') as f:
            for line in f:
                try:
                    # Parse each line as a JSON object
                    json_obj = json.loads(line.strip())  # strip() to remove extra whitespace
                    
                    # Filter by 'language_tag' starting with 'en_'
                    if filter_by_language_tag(json_obj):
                        json_data.append(json_obj)
                
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON in {json_file}: {e}")

# Convert the filtered list of JSON objects into a pandas DataFrame
df = pd.json_normalize(json_data)

# Display the first few rows of the DataFrame
print(df.tail(1))

# Optional: Save the filtered DataFrame to a CSV file
df.to_csv('/kaggle/working/listings_data_filtered.csv', index=False)


## Start from here for only english filtered data

In [ ]:
import pandas as pd

# Define the input directory and file path
input_dir = '/kaggle/input/listing-filtered'
file_path = f'{input_dir}/listings_data_filtered.csv'

# Read the CSV file into a DataFrame
df_english = pd.read_csv(file_path)

# Display the first few rows of the DataFrame to verify the contents
print(df_english.head())


In [ ]:
# Get column names in df_english

# Get the column names of the DataFrame
column_names = df_english.columns

# Print the column names
print(column_names)


In [ ]:
# Loop through the columns and print the column name and its type
for column in df.columns:
    print(f"Column: {column}, Type: {df[column].dtype}")


In [ ]:
import ast

def parse_string_to_object(entry):
    """Safely parse a string entry to a Python object (like a list or dict)."""
    if isinstance(entry, str):
        try:
            # Check if the string looks like a JSON object or list
            if entry.startswith("[") or entry.startswith("{"):
                entry = ast.literal_eval(entry)
        except (ValueError, SyntaxError) as e:
            print(f"Error evaluating string entry: {e}")
            return None
    return entry


def process_standardized_values(entry):
    """Process and return 'standardized_values' from the list of dictionaries."""
    if isinstance(entry, list) and entry and isinstance(entry[0], dict):
        first_item = entry[0]
        if 'standardized_values' in first_item:
            return '| '.join([str(item) for item in first_item['standardized_values']])
    return None


def process_valid_entries(entry):
    """Extract 'value' entries for language tags starting with 'en_'."""
    if isinstance(entry, list) and entry and isinstance(entry[0], dict):
        valid_entries = [
            item['value'] for item in entry
            if isinstance(item, dict) and item.get('language_tag', '').startswith('en_') and 'value' in item
        ]
        if valid_entries:
            return '| '.join([str(entry) for entry in valid_entries])
    return None


def process_all_values(entry):
    """Process and return all 'value' entries from the list of dictionaries."""
    if isinstance(entry, list) and entry and isinstance(entry[0], dict):
        return '| '.join([str(item['value']) for item in entry if isinstance(item, dict) and 'value' in item])
    return None


def process_node_values(entry):
    """Process and return node values with 'node_id' and 'node_name'."""
    if isinstance(entry, list) and entry and isinstance(entry[0], dict):
        first_item = entry[0]
        if 'node_id' in first_item and 'node_name' in first_item:
            return '| '.join(
                [f"nodeid:{item['node_id']},node_name:'{item['node_name']}'" for item in entry if isinstance(item, dict) and 'node_id' in item and 'node_name' in item]
            )
    return None


def process_strings(entry):
    """If it's a list of strings, join them into a single string with '|'."""
    if isinstance(entry, list) and all(isinstance(i, str) for i in entry):
        return '| '.join(entry)
    return None


def process_item_weight(entry):
    """Process the item_weight to return formatted results like wt_gm and wt_pound."""
    result = []

    # Ensure the entry is a list and not empty
    if isinstance(entry, list) and entry:
        for item in entry:
            if isinstance(item, dict):
                # Extract value and unit for grams
                value = item.get('value')  # Top-level value (grams)
                unit = item.get('unit')  # Top-level unit (grams)
                
                # Extract normalized value and normalized unit for pounds
                normalized_value = item.get('normalized_value', {}).get('value')  # Normalized value (pounds)
                normalized_unit = item.get('normalized_value', {}).get('unit')  # Normalized unit (pounds)

                # Now process the value and unit to return a formatted result
                if value is not None and unit:
                    # For grams, use 'wt_gm'
                    result.append(f"wt_gm: {value}")

                if normalized_value is not None and normalized_unit:
                    # For pounds, use 'wt_pound' and format the value to 4 decimal places
                    result.append(f"wt_pound: {normalized_value:.4f}")

    # Return the formatted result as a pipe-separated string
    if result:
        return ' | '.join(result)
    
    return None  # Return None if no valid data found


def process_column_value(entry, column_name):
    """Process the entry based on its type and structure."""
    # Debug the original value before processing
    #print(f"Debug: Original entry before processing for column '{column_name}': {entry}")

    # Parse string to object if necessary
    entry = parse_string_to_object(entry)

    # Early exit for simple strings (non-JSON-like)
    if isinstance(entry, str):
        return entry

    # If processing item_weight column
    if column_name == 'item_weight':  # Directly check the column name
        #print("Calling process_item_weight")
        result = process_item_weight(entry)
        if result:
            return result

    # Process other entries (standardized_values, valid_entries, etc.)
    result = process_standardized_values(entry)
    if result:
        return result

    result = process_valid_entries(entry)
    if result:
        return result

    result = process_all_values(entry)
    if result:
        return result

    result = process_node_values(entry)
    if result:
        return result

    result = process_strings(entry)
    if result:
        return result

    # If no valid processing was done, return None
    return None



# # Example: Mock DataFrame for testing
# df_english = {
#     'brand': ['Amazon Brand - Solimo'],
#     'item_weight': [
#         [{'normalized_value': {'unit': 'pounds', 'value': 0.110231131}, 'unit': 'grams', 'value': 50}]
#     ], 
#     'item_id': ['B07T8PZ94H'],
#     'item_name': ['Amazon Product Name']
# }



In [ ]:
# Columns to process
columns = [
    'brand', 'bullet_point', 'color', 'item_id', 'item_name', 'item_weight',
    'material', 'model_name', 'model_number', 'product_type',
    'main_image_id', 'other_image_id', 'item_keywords', 'country',
    'marketplace', 'domain_name', 'node', 'style',
    'item_dimensions.height.normalized_value.unit',
    'item_dimensions.height.normalized_value.value',
    'item_dimensions.height.unit', 'item_dimensions.height.value',
    'item_dimensions.length.normalized_value.unit',
    'item_dimensions.length.normalized_value.value',
    'item_dimensions.length.unit', 'item_dimensions.length.value',
    'item_dimensions.width.normalized_value.unit',
    'item_dimensions.width.normalized_value.value',
    'item_dimensions.width.unit', 'item_dimensions.width.value',
    'color_code', 'spin_id', '3dmodel_id', 'fabric_type', 'model_year',
    'pattern', 'item_shape', 'finish_type', 'product_description'
]


In [ ]:
def process_dataframe(df):
    return pd.DataFrame({col: df[col].map(lambda entry: process_column_value(entry, col)) for col in df.columns})


In [ ]:
processed_df = process_dataframe(df_english)


In [ ]:
processed_df.to_csv("processed_df_dataset.csv", index=False)

## Start from here for the dataframe with all the listing values: processed_listing_dataset

In [ ]:
# Define the input directory and file path
input_dir_processed = '/kaggle/input/processed-listing-dataset'
file_path_processed = f'{input_dir_processed}/processed_df_dataset.csv'

# Read the CSV file into a DataFrame
df_processed = pd.read_csv(file_path_processed)

In [ ]:
processed_df_new = df_processed

In [ ]:

# Extract weight values
processed_df_new["wt_gm"] = processed_df_new["item_weight"].str.extract(r'wt_gm:\s*([\d.]+)').astype(float)
processed_df_new["wt_pound"] = processed_df_new["item_weight"].str.extract(r'wt_pound:\s*([\d.]+)').astype(float)

# Extract node values
processed_df_new["nodeid"] = processed_df_new["node"].str.extract(r'nodeid:(\d+)')
processed_df_new["nodeid"] = processed_df_new["nodeid"].astype("Int64")  # Nullable integer type

processed_df_new["node_name"] = processed_df_new["node"].str.extract(r"node_name:'(.*?)'")
# Drop original columns if not needed
processed_df_new.drop(columns=["item_weight", "node"], inplace=True)

In [ ]:
processed_df_new.head(5)

In [ ]:
processed_df_new.to_csv("processed_df_dataset_separate_columns.csv", index=False)

## Start frm here with all data in separate columns, except for multi value which is separated by pipe

In [ ]:
# Define the input directory and file path
input_dir_separate_columns = '/kaggle/input/separate-columns'
file_path_processed = f'{input_dir_separate_columns}/processed_df_dataset_separate_columns.csv'

# Read the CSV file into a DataFrame
df_processed_separate_columns = pd.read_csv(file_path_processed)

In [ ]:
df_selected = df_processed_separate_columns[['brand', 'bullet_point', 'color', 'item_id', 'model_name', 'model_number', 'product_type', 'main_image_id', 'other_image_id', 'item_keywords', 'style']]


In [ ]:
df_selected.to_csv("df_selected.csv", index=False)

# start from here for selected columns only

In [78]:
# Define the input directory and file path
input_dir_selected_columns = '/kaggle/input/selected-columns'
file_path_selected = f'{input_dir_selected_columns}/df_selected.csv'

# Read the CSV file into a DataFrame
df_processed_selected_columns = pd.read_csv(file_path_selected)

In [79]:
df_processed_selected_columns.head(5)

,brand,bullet_point,color,item_id,model_name,model_number,product_type,main_image_id,other_image_id,item_keywords,style
0,Amazon Brand - Solimo,"Snug fit for Samsung Galaxy A10s, with perfect...",multi-colored,B0854774MY,Samsung Galaxy A10s,UV10797-SL13059,CELLULAR_PHONE_CASE,71owAzvPFuL,61+woWTqkwL| 61SE4RTPjdL,Back Cover| Designer Case| Designer Chinnese Y...,NaN
1,Ravenna Home,NaN,Light Grey,B07KRBPKT6,NaN,25030020-01,BENCH,A114ZfnM0dL,NaN,NaN,Raconteur Fog
2,365 Everyday Value,Brought to you by Whole Foods Market. The pac...,NaN,B086HH8YDJ,NaN,NaN,GROCERY,816oDloCY4L,51u3Lkk-R+L| 71jPhuf7zsL| 71vrYGQp3zL| 81b11AK...,NaN,NaN
3,Amazon Brand - Solimo,"Snug fit for Oppo A7, with perfect cut-outs fo...",multi-colored,B0856BGC3J,Oppo A7,UV10766-SL40397,CELLULAR_PHONE_CASE,71GmYaibrUL,61+woWTqkwL| 61SE4RTPjdL,Back Cover| Designer Case| Designer Peacock Fe...,NaN
4,Engine 2,Brought to you by Whole Foods Market.| No adde...,NaN,B074H73842,NaN,NaN,GROCERY,91w54-A-WFL,81XfutyF3UL| 61c2N2TghAL| 71aj6Lyz0TL| 81GspmL...,Placeholder| Placeholder| Placeholder| Placeho...,NaN


# some EDA for selected columns

In [86]:
# Check for null values in the entire DataFrame
null_values = df_processed_selected_columns.isnull().sum()

# Print the number of null values in each column
print(null_values)


brand                30
bullet_point       7753
color             21014
item_id               0
model_name        44846
model_number      16291
product_type          0
main_image_id       434
other_image_id     6430
item_keywords     10987
style             85122
dtype: int64


In [89]:
# Calculate the percentage of null values in each column
null_percentage = df_processed_selected_columns.isnull().mean() * 100

# Print the percentage of null values
print(null_percentage)


brand              0.026074
bullet_point       6.738516
color             18.264308
item_id            0.000000
model_name        38.977880
model_number      14.159315
product_type       0.000000
main_image_id      0.377211
other_image_id     5.588632
item_keywords      9.549346
style             73.983747
dtype: float64


##item_name, bullet_point, item_keywords ,brand, color, material ,style, product_type, model_name, model_number columns can be embedded 